# FHRPY — CTG viewer & analysis demo

Analyse a full cardiotocography (CTG) recording, then compare the **expert
labels** with the **FHRPY methods** (false-signal detection and WMFB baseline),
all inline under **VSCode** and **Google Colab**.

> **Run the setup cell below first.** It makes `fhrpy` importable on its own —
> no manual `pip install` from a checkout of
> [data-coeur/FHRPY](https://github.com/data-coeur/FHRPY).

In [ ]:
# === Setup — run this cell first (makes the notebook self-contained) ===
# If you just pulled new code, RESTART THE KERNEL first: Python caches modules,
# so a previously-imported `fhrpy` (e.g. an older dataset list) stays in memory.
import sys, pathlib
# Prefer the local checkout (so the bundled datasets/examples are the current
# ones) by putting the repo root FIRST on sys.path.
_root = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
              if (p / "fhrpy" / "__init__.py").exists()), None)
if _root and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
try:
    import fhrpy
except ModuleNotFoundError:
    # Standalone (e.g. Google Colab): install the released package from GitHub.
    # The `main` branch is public, so NO token is required.
    %pip install -q "fhrpy[viewer] @ git+https://github.com/data-coeur/FHRPY.git@main"
    import fhrpy

import fhrpy.datasets as _ds
print("fhrpy", fhrpy.__version__, "from", fhrpy.__file__)
if not any(e["name"].startswith("ctg_") for e in _ds.list_examples()):
    print("\n⚠️  A stale `fhrpy` is cached — RESTART THE KERNEL and re-run this cell.")

In [ ]:
import fhrpy.datasets as ds
from fhrpy.viewer import FHRViewer, link_scroll
from IPython.display import display, HTML

# Bundled FHRMA examples (all start at 00h00; expert labels in companion .fhrh):
for e in ds.list_examples():
    print(f"{e['name']:16s} {e['category']:9s} {e['duration_min']:6.1f} min  -  {e['description']}")

## 1. Analyse a complete CTG

By default we open one of the FHRMA **Examples** recordings (a full Doppler CTG)
and run the **whole FHRPY pipeline**: false-signal detection & removal → WMFB
baseline → accelerations / decelerations. The blue **Expulsion** line is the
protected `£Expulsion` marker (its `£` prefix makes it non-editable), read from
the recording — no need to pass the expulsion time.

Change `EXAMPLE` to any `ctg_*` name listed above. Use the toolbar to toggle the
baseline / contraction / false-signal zones, switch 1↔3 cm/min, measure, or
print to PDF.

In [ ]:
# Pick the CTG to analyse: one line is uncommented (the default), the other
# FHRMA "Examples" recordings are alternatives — just move the comment.
EXAMPLE = "ctg_example_01"   # Doppler and Scalp Stage2 at min 215 (expulsion min 215)
# EXAMPLE = "ctg_example_02"   # Doppler Pattern D Stage 2 at min 275 (expulsion min 275)
# EXAMPLE = "ctg_example_03"   # Doppler and Scalp Stage 2 at min 502 (expulsion min 502)
# EXAMPLE = "ctg_example_04"   # Doppler stage 2 at min 360 (expulsion min 360)
# EXAMPLE = "ctg_example_05"   # Doppler No real baseline Stage 2 at min 330 (expulsion min 330)
# EXAMPLE = "ctg_example_06"   # Doppler Stage 2 at min 247 (expulsion min 247)
# EXAMPLE = "ctg_example_07"   # Doppler Long FS Stage 2 at min 475 (expulsion min 475)
# EXAMPLE = "ctg_example_08"   # Doppler Stage 2 at min 318 (expulsion min 318)
# EXAMPLE = "ctg_example_09"   # Doppler Stage 2 at min 497 (expulsion min 497)
# EXAMPLE = "ctg_example_10"   # Doppler Stage 2 at min 580 (expulsion min 580)
# EXAMPLE = "ctg_example_11"   # Doppler Stage 2 at min 365 (expulsion min 365)

ds.load_example(EXAMPLE, source="method", height=480)

## 2. Expert labels vs prediction — **false signals**

Here we use a **training** record (which carries the expert ground truth) and
show, side by side, the **expert** false-signal episodes and the **FHRPY**
detector. `link_scroll(...)` keeps both windows at the **same instant**: scroll,
page or wheel-scroll either one and the other follows — so label/prediction
differences line up.

> **False-signal detection requirements.** Doppler detection expects `FHR1` = the
> **Doppler** channel and `FHR2` = the **scalp** (direct fetal ECG) channel. The
> maternal-heart-rate channel (`MHR`) must be **time-aligned** to the FHR first:
> on Philips monitors the maternal pulse lags by about **12.5 s** with the SpO2
> **oximeter**, or about **5 s** when derived from the **TOCO/belt** — shift `MHR`
> by that delay before running detection.

In [ ]:
NAME = "fs_dopmhr_01"   # a DopMHR training record with expert FS labels
fs_label = ds.load_example(NAME, source="expert", channels=["FHR1", "MHR"], height=320)
fs_pred  = ds.load_example(NAME, source="method", channels=["FHR1", "MHR"], height=320)

display(HTML("<b>Expert false-signal labels</b>")); display(fs_label)
display(HTML("<b>FHRPY false-signal detection</b>")); display(fs_pred)
link_scroll(fs_label, fs_pred)   # both viewers stay at the same instant

## 3. Expert labels vs prediction — **baseline / accel / decel (Ldb)**

Same idea on a **morphology** training record: the **expert** baseline +
acceleration (green) / deceleration (red) zones vs the **FHRPY WMFB** baseline +
detected accel/decel — scroll-synchronised.

In [ ]:
NAME = "morpho_train21"   # a morphology training record with expert labels
bl_label = ds.load_example(NAME, source="expert", height=320)
bl_pred  = ds.load_example(NAME, source="method", height=320)

display(HTML("<b>Expert baseline + accel/decel</b>")); display(bl_label)
display(HTML("<b>FHRPY WMFB baseline + accel/decel</b>")); display(bl_pred)
link_scroll(bl_label, bl_pred)

## 4. Going further — head-less methods, dynamic control, export

**(a) Run the methods without any UI**, inspect the numbers, then use them.

In [ ]:
from fhrpy.io import read_fhr
from fhrpy.baseline import analyze, compute_features
from fhrpy.falsesignal import detect_false_signals

rec = read_fhr(ds.example_path("ctg_example_01"))
ma = analyze(rec)                                  # WMFB baseline + morphology + contractions
fs = detect_false_signals(rec, kind="doppler")     # false-signal probabilities
print("baseline pts:", len(ma["baseline"]),
      "| acc:", len(ma["accelerations"]), "| dec:", len(ma["decelerations"]),
      "| contractions:", len(ma["contractions"]))
print("false-signal episodes:", len(fs["segments"]),
      "| P(false) max:", round(float(fs["prob"].max()), 3))

**(b) Feature synthesis.** `compute_features` returns the standard CTG
features (English names) — baseline / time-in-band, accel & decel morphology,
deceleration types, contractions and short-/long-term variability.

In [ ]:
feats = compute_features(rec)
print(len(feats), "features. A few standard ones:")
for k in ("baseline_mean", "fhr_time_below_110_percent",
          "acc_count", "dec_count", "dec_surface_total",
          "dec_early_count", "dec_late_count", "dec_variable_count",
          "contraction_count", "dec_to_contraction_ratio",
          "stv_msd", "ltv_delta_mean"):
    print(f"  {k:34s} {round(feats[k], 2)}")

**(b) Drive the viewer dynamically from Python** and listen to its events.

In [ ]:
v = ds.load_example("ctg_example_01", source="method", height=440)
display(v)

v.set_scale(3)                  # 3 cm/min paper speed
v.set_safe_zone(110, 150)       # move the central "safe" band
v.set_range(50, 210)            # top-grid bounds
v.scroll_to(0.5)                # jump to the middle
v.set_false_signals_visible(True)
v.on("scroll", lambda d: print("scrolled to", round(d.get("time", 0) / 60, 1), "min"))
# also: set_contractions_visible, set_zones_visible, set_interpolate,
#       set_channel_visible, set_markers / getMarkers, print()

**(c) Export a standalone offline HTML** with exactly the same parameters
(no server, no PHP). The toolbar printer button gives a multi-page **A4-landscape
PDF** (≈ 1 cm/min, 20 bpm/cm).

In [ ]:
v = ds.load_example(
    "ctg_example_01", source="method",
    channels=["FHR1", "MHR"], scale=1, height=460,
    rcf_min=50, rcf_max=210,      # configurable top-grid bounds
    safe_min=110, safe_max=160,   # configurable central safe band
    interpolate=True,
)
v.to_html("ctg_example.html")     # offline page with exactly these parameters
print("wrote ctg_example.html")